# 📊 NLLB Training Data — Exploration & Demo

Notebook này demo:
1. Load và phân tích dataset
2. Visualize chunk distribution
3. So sánh baseline vs fine-tuned translation
4. Lỗi dịch phổ biến

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_theme(style='whitegrid')
print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
def load_jsonl(path):
    data = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load processed data
try:
    train_data = load_jsonl('../data/processed/train.jsonl')
    valid_data = load_jsonl('../data/processed/valid.jsonl')
    test_data  = load_jsonl('../data/processed/test.jsonl')
    print(f'Train: {len(train_data):,}')
    print(f'Valid: {len(valid_data):,}')
    print(f'Test:  {len(test_data):,}')
except FileNotFoundError:
    print('Run scripts/prepare_data.py first!')
    train_data = []

In [ ]:
# Convert to DataFrame
if train_data:
    df = pd.DataFrame(train_data)
    print(df[['src_lang', 'tgt_lang', 'src', 'tgt']].head(5))
    print('\nLang pair distribution:')
    print(df.groupby(['src_lang', 'tgt_lang']).size())

## 2. Token Length Distribution

In [ ]:
if train_data:
    df['src_len'] = df['src'].apply(lambda x: len(x.split()))
    df['tgt_len'] = df['tgt'].apply(lambda x: len(x.split()))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['src_len'].hist(bins=40, ax=axes[0], color='steelblue')
    axes[0].set_title('Source Token Length')
    axes[0].set_xlabel('# tokens')

    df['tgt_len'].hist(bins=40, ax=axes[1], color='coral')
    axes[1].set_title('Target Token Length')
    axes[1].set_xlabel('# tokens')

    plt.tight_layout()
    plt.show()
    print(df[['src_len', 'tgt_len']].describe())

## 3. Chunk Type Distribution

In [ ]:
if train_data and 'src_chunks' in train_data[0]:
    chunk_types = []
    for item in train_data[:5000]:  # sample 5k
        chunks = item.get('src_chunks', {}).get('chunks', [])
        for c in chunks:
            chunk_types.append(c['type'])

    ct_counter = Counter(chunk_types)
    labels, counts = zip(*ct_counter.most_common())

    plt.figure(figsize=(8, 4))
    plt.bar(labels, counts, color='steelblue')
    plt.title('Chunk Type Distribution (sample 5k sentences)')
    plt.xlabel('Chunk Type')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Run scripts/annotate_chunks.py first!')

## 4. Demo: Chunker

In [ ]:
from src.chunker import get_chunker, assign_context_weights

# English example
try:
    en_chunker = get_chunker('en')
    sent = 'The deployment is scheduled for tomorrow morning in the production environment.'
    chunked = en_chunker.chunk(sent)
    chunked = assign_context_weights(chunked)

    print(f'Input: {sent}\n')
    print(f'{"Chunk":<40} {"Type":<6} {"Role":<8} {"Pos":<6} {"Weight"}')
    print('-' * 70)
    for c in chunked.chunks:
        print(f'{c.text:<40} {c.chunk_type.value:<6} {c.dep_role:<8} {c.position_ratio:.2f}   {c.context_weight:.2f}')
except OSError as e:
    print(f'Install spaCy model: {e}')

In [ ]:
# Japanese example
try:
    ja_chunker = get_chunker('ja')
    ja_sent = '会議は午後3時に始まり、プロジェクトのデプロイについて話し合います。'
    ja_chunked = ja_chunker.chunk(ja_sent)
    ja_chunked = assign_context_weights(ja_chunked)

    print(f'Input: {ja_sent}\n')
    for c in ja_chunked.chunks:
        print(f'  [{c.chunk_type.value}] {c.text!r} (role={c.dep_role}, w={c.context_weight:.2f})')
except OSError as e:
    print(f'Install spaCy model: python -m spacy download ja_core_news_sm')

## 5. Demo: Contextual Translation

In [ ]:
# Load translator (requires model)
try:
    from src.translator import ContextualTranslator

    # Use baseline NLLB (or fine-tuned: change path)
    translator = ContextualTranslator(
        model_path='facebook/nllb-200-distilled-600M',
        use_chunk_prefix=True,
    )

    examples = [
        ('The API server is down and we need to deploy a hotfix immediately.', 'en', 'vi'),
        ('会議は明日の午後3時から5時まで予定されています。', 'ja', 'vi'),
        ('スプリントのレビューはいつですか？', 'ja', 'en'),
    ]

    for text, src, tgt in examples:
        details = translator.translate_with_details(text, src, tgt)
        print(f'\nInput ({src}): {text}')
        print(f'Output ({tgt}): {details["final_translation"]}')
        print('Chunks:')
        for c in details['chunks']:
            print(f'  [{c["type"]}] {c["chunk"]!r} → {c["translated"]!r} (w={c["context_weight"]:.2f})')
except Exception as e:
    print(f'Error: {e}\nDownload model first: make download-model')

## 6. Context Weight Heatmap

In [ ]:
# Visualize context weights cho 1 câu ví dụ
import numpy as np

try:
    from src.chunker import get_chunker, assign_context_weights

    chunker = get_chunker('en')
    sent = 'The development team needs to review the security vulnerability before the release.'
    chunked = assign_context_weights(chunker.chunk(sent))

    chunk_labels = [f'{c.text[:15]}...\n[{c.chunk_type.value}]' if len(c.text) > 15 else f'{c.text}\n[{c.chunk_type.value}]' for c in chunked.chunks]
    weights = [c.context_weight for c in chunked.chunks]

    fig, ax = plt.subplots(figsize=(max(8, len(weights) * 1.5), 2.5))
    im = ax.imshow([weights], cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(weights)))
    ax.set_xticklabels(chunk_labels, fontsize=8, rotation=15, ha='right')
    ax.set_yticks([])
    ax.set_title('Context Weights per Chunk', pad=10)
    plt.colorbar(im, ax=ax, label='weight')
    for i, w in enumerate(weights):
        ax.text(i, 0, f'{w:.2f}', ha='center', va='center', fontsize=9, color='black' if w < 0.7 else 'white')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Install spaCy: {e}')